# 04 - Multivariate tail dependence

Research question: are joint tail events more persistent than marginal exceedances in a coupled multivariate system?

Data dependencies: none. This notebook uses synthetic coupled-map data.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dynsys_econometrics.extremes import extract_threshold_exceedances
from dynsys_econometrics.simulation import simulate_coupled_logistic_maps


In [ ]:
n_steps = 5000
coupled = simulate_coupled_logistic_maps(n_steps, n_series=3, coupling=0.15, seed=19)
frame = pd.DataFrame(
    {
        "series_a": coupled[:, 0],
        "series_b": 0.6 * coupled[:, 1] + 0.4 * coupled[:, 2],
    }
)

rows = []
for quantile in [0.90, 0.95, 0.975, 0.99]:
    threshold_a = float(frame["series_a"].quantile(quantile))
    threshold_b = float(frame["series_b"].quantile(quantile))
    exceed_a = extract_threshold_exceedances(frame["series_a"].to_numpy(), threshold_a)
    exceed_b = extract_threshold_exceedances(frame["series_b"].to_numpy(), threshold_b)
    joint_hits = np.intersect1d(exceed_a.indices, exceed_b.indices)
    rows.append(
        {
            "quantile": quantile,
            "marginal_a": exceed_a.values.size / len(frame),
            "marginal_b": exceed_b.values.size / len(frame),
            "joint": joint_hits.size / len(frame),
        }
    )

tail_summary = pd.DataFrame(rows)
tail_summary


In [ ]:
output_dir = Path("article/figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tail_summary["quantile"], tail_summary["marginal_a"], marker="o", label="marginal a")
ax.plot(tail_summary["quantile"], tail_summary["marginal_b"], marker="s", label="marginal b")
ax.plot(tail_summary["quantile"], tail_summary["joint"], marker="^", label="joint")
ax.set_title("Tail probabilities across thresholds")
ax.set_xlabel("quantile")
ax.set_ylabel("tail probability")
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "04_multivariate_tail_dependence.png", dpi=160)
plt.close(fig)
str(output_dir / "04_multivariate_tail_dependence.png")
